# Continue Ettin: benchmark prefixes, optionally refine with Matryoshka

 Self-contained A100 / VS Code Colab notebook. Load the successful pilot's validation-best checkpoint,
 evaluate **128 / 256 / 384 dimensions**, and optionally train nested prefixes in a separate Drive run.
 All code is inline. Notebook 02 does not need to be open or executing.

 Default behavior is evaluation-only. No teacher model is loaded, no pilot training is repeated,
 and the source run is read-only. Keep the source checkpoint/data/cache folders on Drive.

In [1]:
# Install into this notebook's kernel. Keep Colab's CUDA-enabled PyTorch.
# Restart the kernel after this cell if it already imported different versions.
%pip install -q transformers==5.1.0 datasets==4.5.0 accelerate==1.12.0 sentence-transformers==5.2.2 safetensors

In [2]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = '/content/hf-cache' if os.path.isdir('/content') else '/tmp/encoder-hf-cache'
import copy, gc, hashlib, json, math, random, shutil, time, uuid
from pathlib import Path
from collections import defaultdict
from contextlib import nullcontext
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file, save_file
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')
DEVICE = 'cuda'
assert torch.cuda.is_available(), 'Select an A100 GPU runtime in Colab.'
assert torch.cuda.is_bf16_supported(), 'This notebook uses BF16 on a supported GPU.'
print(torch.cuda.get_device_name(), 'PyTorch:', torch.__version__)
torch.set_float32_matmul_precision('high')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NVIDIA A100-SXM4-40GB PyTorch: 2.11.0+cu128


## Choose the source checkpoint and new destination

In [3]:
# Source is read-only. A separate destination preserves the successful pilot.
SOURCE_RUN = Path('/content/drive/MyDrive/encoder_experiments/ettin-pilot-01')
SOURCE_STAGE = 'student'
SOURCE_CHOICE = 'best'  # 'best', 'latest', or an integer optimizer step (e.g. 400)
RUN_NAME = 'ettin-prefix-eval-01'
ROOT = Path('/content/drive/MyDrive/encoder_experiments') / RUN_NAME
assert ROOT.resolve() != SOURCE_RUN.resolve(), 'Use a separate destination run.'
LOCAL = Path('/content/encoder-work') / RUN_NAME
ROOT.mkdir(parents=True, exist_ok=True); LOCAL.mkdir(parents=True, exist_ok=True)

RUN_MATRYOSHKA = False  # Default: benchmark existing prefixes, with NO training.
DIMS = [128, 256, 384]
REFINEMENT = dict(steps=300, lr=1e-5, checkpoint_every=50, eval_every=50,
                  log_every=10, warmup_fraction=0.1)

def digest(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def text_id(s):
    return hashlib.sha256(' '.join(s.split()).casefold().encode()).hexdigest()
def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
    temp.write_text(json.dumps(obj,indent=2,ensure_ascii=False));os.replace(temp,path)
def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for part in iter(lambda:f.read(8*1024*1024),b''):h.update(part)
    return h.hexdigest()

source_contract=json.loads((SOURCE_RUN/'config.json').read_text())
assert source_contract['config']['experiment']=='ettin'
CFG=copy.deepcopy(source_contract['config'])
CFG.update(experiment='ettin-prefix-continuation',run_matryoshka=RUN_MATRYOSHKA,
           matryoshka_steps=REFINEMENT['steps'],student_lr=REFINEMENT['lr'],
           checkpoint_every=REFINEMENT['checkpoint_every'],eval_every=REFINEMENT['eval_every'],
           log_every=REFINEMENT['log_every'],warmup_fraction=REFINEMENT['warmup_fraction'])
GTE=source_contract['gte'];ETTIN=source_contract['ettin'];NANO=source_contract['nano']
seed_everything(CFG['seed'])
print('Source:',SOURCE_RUN,'\nDestination:',ROOT,'\nMatryoshka enabled:',RUN_MATRYOSHKA)

Source: /content/drive/MyDrive/encoder_experiments/ettin-pilot-01 
Destination: /content/drive/MyDrive/encoder_experiments/ettin-prefix-eval-01 
Matryoshka enabled: False


## Verify and lock the baseline
 The default reads `best_step` from the latest valid pilot checkpoint; the completed pilot selected step 400.
 The selected generation and checksum are locked in the destination, so resuming cannot silently switch sources.
 Use an integer for `SOURCE_CHOICE` to select another retained checkpoint explicitly.

In [4]:
# Only load your own training checkpoints. Verify the source manifest and bytes first.
def verified_source_generations():
    parent=SOURCE_RUN/'checkpoints'/SOURCE_STAGE
    candidates=sorted((p for p in parent.iterdir() if not p.name.startswith('.') and
                       (p/'COMMITTED.json').exists()),reverse=True)
    expected=digest(source_contract)
    for path in candidates:
        meta=json.loads((path/'COMMITTED.json').read_text())
        if meta['contract']!=expected:
            raise RuntimeError(f'Source checkpoint belongs to another config: {path}')
        yield path,meta

def read_source(path,meta):
    local=LOCAL/'source-checkpoint.pt'
    shutil.copyfile(path/'state.pt',local)
    if file_sha(local)!=meta['sha256']:
        raise ValueError(f'Source checkpoint checksum mismatch: {path}')
    state=torch.load(local,map_location='cpu',weights_only=False)
    assert state['contract']==digest(source_contract) and state['family']=='ettin'
    assert state['step']==meta['step'] and state['stage']==SOURCE_STAGE
    return state

def select_source_checkpoint():
    lock=ROOT/'source.json'
    if lock.exists():
        selected=json.loads(lock.read_text())
        assert selected['run']==str(SOURCE_RUN) and selected['stage']==SOURCE_STAGE
        assert selected['choice']==SOURCE_CHOICE, 'Source selection changed: choose a new RUN_NAME.'
        path=Path(selected['checkpoint'])
        meta=json.loads((path/'COMMITTED.json').read_text())
        assert meta['sha256']==selected['sha256'] and meta['contract']==digest(source_contract)
        return selected,read_source(path,meta)
    generations=list(verified_source_generations())
    if not generations:raise FileNotFoundError('No completed source checkpoints. Finish notebook 02 first.')
    latest=None
    for path,meta in generations:
        try:
            latest=read_source(path,meta);break
        except (ValueError,OSError,EOFError) as error:
            print('Skipping unreadable generation:',path,error)
    if latest is None:raise RuntimeError('No readable source checkpoint remains.')
    step=latest['best_step'] if SOURCE_CHOICE=='best' else latest['step'] if SOURCE_CHOICE=='latest' else int(SOURCE_CHOICE)
    if latest['step']!=step:
        del latest
        choices=[(p,x) for p,x in generations if x['step']==step]
        if not choices:raise FileNotFoundError(f'Source step {step} was not retained.')
        path,meta=choices[0];latest=read_source(path,meta)
    selected=dict(run=str(SOURCE_RUN),stage=SOURCE_STAGE,choice=SOURCE_CHOICE,step=step,
                  checkpoint=str(path),sha256=meta['sha256'])
    write_json(lock,selected)
    return selected,latest

source_info,source_state=select_source_checkpoint()
DATA_ID=source_state['data_id']
contract=dict(config=CFG,gte=GTE,ettin=ETTIN,nano=NANO,source=source_info,
              dimensions=DIMS,refinement=REFINEMENT,selection='mean_validation_prefix_mrr',implementation=1)
CONTRACT_ID=digest(contract)
if (ROOT/'config.json').exists():
    assert json.loads((ROOT/'config.json').read_text())==contract, 'Settings changed: choose a new RUN_NAME.'
else:write_json(ROOT/'config.json',contract)
print('Selected source step:',source_info['step'],'; checkpoint:',source_info['checkpoint'])
print('Source identity is now locked for this destination run.')

Selected source step: 400 ; checkpoint: /content/drive/MyDrive/encoder_experiments/ettin-pilot-01/checkpoints/student/00000400-88ddf87f
Source identity is now locked for this destination run.


In [5]:
def mean_pool(hidden, mask):
    mask = mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden.float()*mask).sum(1) / mask.sum(1).clamp_min(1)

# Input adapters belong to the model, so evaluation, training and export use the same tokenizer/pooling.
class EmbeddingModel(nn.Module):
    def __init__(self, backbone, tokenizer, output_dim, family, spec):
        super().__init__()
        self.backbone, self.tokenizer = backbone, tokenizer
        self.output_dim, self.family, self.spec = output_dim, family, copy.deepcopy(spec)
        self.max_length = CFG['max_length']
        # Ettin's alignment head is training-only. Retrieval always uses its native 384/512 dimensions.
        self.align = nn.Identity() if output_dim == 768 else nn.Linear(output_dim,768,bias=False)
    def forward(self, texts):
        tokens = self.tokenizer(texts, padding=True, truncation=True, max_length=self.max_length,
                                return_tensors='pt').to(next(self.parameters()).device)
        if self.family == 'neobert':
            h = self.backbone(tokens['input_ids'], tokens['attention_mask'])
        else:
            h = self.backbone(**tokens).last_hidden_state
        return mean_pool(h, tokens['attention_mask'])

def count_parameters(model):
    return dict(total=sum(p.numel() for p in model.parameters()),
                backbone=sum(p.numel() for p in model.backbone.parameters()),
                training_only_alignment=sum(p.numel() for p in model.align.parameters()))

In [6]:
# Reconstruct from the saved architecture, then STRICTLY load trained weights including alignment head.
spec=copy.deepcopy(source_state['spec'])
model_type=spec.pop('model_type')
assert model_type=='modernbert'
backbone_config=AutoConfig.for_model(model_type,**spec)
backbone=AutoModel.from_config(backbone_config,attn_implementation='sdpa')
if CFG['gradient_checkpointing']:
    backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
tokenizer=AutoTokenizer.from_pretrained(ETTIN['id'],revision=ETTIN['revision'])
student=EmbeddingModel(backbone,tokenizer,backbone_config.hidden_size,'ettin',source_state['spec'])
student.load_state_dict(source_state['model'],strict=True)
student.to(DEVICE).eval()
assert DIMS==sorted(set(DIMS)) and all(0<d<=student.output_dim for d in DIMS)
assert student.output_dim in DIMS, 'Include full width as the baseline profile.'
del source_state,backbone
# The small CPU copy is the immutable baseline for rerunning the optional training cell.
baseline_weights={k:v.detach().cpu().clone() for k,v in student.state_dict().items()}
print(count_parameters(student))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


{'total': 32178048, 'backbone': 31883136, 'training_only_alignment': 294912}


## Benchmark natural truncation before changing weights
 These are the same pinned NanoMSMARCO and NanoHotpotQA datasets and sequence-length cap as the pilot.
 Each model encodes once; all prefix scores use the same vectors and candidate pools. Nano results are reporting-only,
 not the criterion for choosing a training checkpoint. Each dataset has 50 queries, so interpret small changes cautiously.

In [7]:
# A pinned, labeled retrieval benchmark. Never used for optimization or early stopping.
BENCHMARKS = {}
def load_benchmarks():
    if BENCHMARKS:
        return BENCHMARKS
    for name in ('NanoMSMARCO', 'NanoHotpotQA'):
        parts = {part: load_dataset(NANO['id'], part, split=name, revision=NANO['revision'])
                 for part in ('queries', 'corpus', 'qrels')}
        queries = {str(r['_id']): r['text'] for r in parts['queries'] if r['text'].strip()}
        corpus = {str(r['_id']): r['text'] for r in parts['corpus'] if r['text'].strip()}
        relevant = defaultdict(set)
        for r in parts['qrels']:
            ids = r['corpus-id'] if isinstance(r['corpus-id'], list) else [r['corpus-id']]
            relevant[str(r['query-id'])].update(str(x) for x in ids)
        BENCHMARKS[name] = dict(queries=queries, corpus=corpus, relevant=dict(relevant))
    return BENCHMARKS

@torch.inference_mode()
def encode_texts(model, texts, batch_size=None, dim=None):
    model.eval()
    result = []
    for start in range(0, len(texts), batch_size or CFG['teacher_batch_size']):
        with torch.autocast('cuda', dtype=torch.bfloat16):
            z = model(texts[start:start+(batch_size or CFG['teacher_batch_size'])])
        z = z.float().cpu()
        assert torch.isfinite(z).all(), 'Non-finite embeddings'
        result.append(F.normalize(z[:, :dim], dim=-1) if dim else F.normalize(z, dim=-1))
    return torch.cat(result)

def retrieval_metrics(q, p, relevant, query_ids, corpus_ids):
    scores = q @ p.T
    ranking = scores.argsort(dim=1, descending=True)
    sums = defaultdict(float)
    for i, qid in enumerate(query_ids):
        truth = relevant[qid]
        top = [corpus_ids[j] for j in ranking[i, :10].tolist()]
        hits = [int(x in truth) for x in top]
        sums['hit@1'] += hits[0]
        sums['hit@5'] += bool(sum(hits[:5]))
        sums['recall@10'] += sum(hits) / len(truth)
        sums['mrr@10'] += next((1/(j+1) for j,h in enumerate(hits) if h), 0)
        dcg = sum(h/math.log2(j+2) for j,h in enumerate(hits))
        ideal = sum(1/math.log2(j+2) for j in range(min(len(truth), 10)))
        sums['ndcg@10'] += dcg / ideal
    return {k: v/len(query_ids) for k,v in sums.items()}

def benchmark(model, label, dims=None):
    results = {}
    for name, b in load_benchmarks().items():
        corpus_ids = list(b['corpus'])
        query_ids = [qid for qid in b['queries'] if b['relevant'].get(qid)]
        assert all(b['relevant'][qid] <= set(corpus_ids) for qid in query_ids)
        q = encode_texts(model, [b['queries'][k] for k in query_ids])
        p = encode_texts(model, list(b['corpus'].values()))
        for dim in dims or [q.shape[1]]:
            assert 0 < dim <= q.shape[1]
            metrics = retrieval_metrics(F.normalize(q[:,:dim], dim=-1), F.normalize(p[:,:dim], dim=-1),
                                        b['relevant'], query_ids, corpus_ids)
            results[f'{name}/{dim}'] = metrics
            print(label, name, dim, {k: round(v,4) for k,v in metrics.items()}, flush=True)
    write_json(ROOT/'evaluations'/f'{label}.json', dict(label=label, contract=CONTRACT_ID,
               benchmark=NANO, max_length=CFG['max_length'], results=results))
    return results

In [8]:
@torch.inference_mode()
def check_padding_invariance(model):
    model.eval()
    short = 'Reset my password.'
    long = 'A substantially longer sentence about weather, gardening, cooking and unrelated household tasks.'
    a = encode_texts(model, [short])
    b = encode_texts(model, [short, long])[:1]
    cosine = F.cosine_similarity(a,b).item()
    print('Single vs padded batch cosine:', cosine)
    assert cosine > 0.999, 'Padding/batching changed the embedding unexpectedly.'

In [9]:
check_padding_invariance(student)

Single vs padded batch cosine: 0.9999803304672241


In [10]:
def compare_prefixes(results, reference=None):
    print('Dataset         dim    hit@1   hit@5   NDCG@10  vs full   bytes/vector FP32')
    for name in load_benchmarks():
        full=results[f'{name}/{student.output_dim}']['ndcg@10']
        for dim in DIMS:
            row=results[f'{name}/{dim}'];delta=row['ndcg@10']-full
            print(f"{name:15} {dim:4}  {row['hit@1']:7.1%} {row['hit@5']:7.1%}  "
                  f"{row['ndcg@10']:7.4f} {delta:+8.4f} {dim*4:10}")
            if reference is not None:
                print(f"  change from original {dim}D: NDCG@10 {row['ndcg@10']-reference[f'{name}/{dim}']['ndcg@10']:+.4f}")
    print('Truncation reduces vector storage and similarity-search work; it does not reduce backbone encoding cost.')

baseline_results=benchmark(student,'source-prefixes',dims=DIMS)
compare_prefixes(baseline_results)
write_json(ROOT/'evaluations'/'baseline-summary.json',dict(source=source_info,dimensions=DIMS,
           results=baseline_results,vector_bytes_fp32={str(d):4*d for d in DIMS}))

source-prefixes NanoMSMARCO 128 {'hit@1': 0.16, 'hit@5': 0.38, 'recall@10': 0.5, 'mrr@10': 0.2627, 'ndcg@10': 0.3191}
source-prefixes NanoMSMARCO 256 {'hit@1': 0.12, 'hit@5': 0.48, 'recall@10': 0.56, 'mrr@10': 0.2749, 'ndcg@10': 0.3442}
source-prefixes NanoMSMARCO 384 {'hit@1': 0.24, 'hit@5': 0.5, 'recall@10': 0.6, 'mrr@10': 0.366, 'ndcg@10': 0.4225}
source-prefixes NanoHotpotQA 128 {'hit@1': 0.52, 'hit@5': 0.8, 'recall@10': 0.55, 'mrr@10': 0.6319, 'ndcg@10': 0.4888}
source-prefixes NanoHotpotQA 256 {'hit@1': 0.6, 'hit@5': 0.82, 'recall@10': 0.56, 'mrr@10': 0.6892, 'ndcg@10': 0.5274}
source-prefixes NanoHotpotQA 384 {'hit@1': 0.6, 'hit@5': 0.82, 'recall@10': 0.58, 'mrr@10': 0.6897, 'ndcg@10': 0.5343}
Dataset         dim    hit@1   hit@5   NDCG@10  vs full   bytes/vector FP32
NanoMSMARCO      128    16.0%   38.0%   0.3191  -0.1034        512
NanoMSMARCO      256    12.0%   48.0%   0.3442  -0.0782       1024
NanoMSMARCO      384    24.0%   50.0%   0.4225  +0.0000       1536
NanoHotpotQA 

## Optional Matryoshka continuation
 Set `RUN_MATRYOSHKA=True` with a **new RUN_NAME**, then run from the top. Defaults: 300 updates at 1e-5.
 Only this optional branch reads the source data and cached targets. The alignment head is restored from the training
 checkpoint; inference exports intentionally omit it and cannot substitute for that checkpoint here.

 Ranking loss averages all configured prefixes; teacher alignment remains full-vector. Checkpoint selection uses
 mean reciprocal rank averaged over prefixes on the full validation split. Step zero is retained and can win if
 training does not improve validation. Existing optimizer/RNG/sampler checkpoints resume this branch after interruption.

In [11]:
# Optional fine-tuning reuses the EXACT pilot data and existing GTE cache; no teacher is loaded.
if RUN_MATRYOSHKA:
    corpus=json.loads((SOURCE_RUN/'pairs.json').read_text())
    assert digest(corpus)==DATA_ID, 'Source pairs no longer match the selected checkpoint.'
    texts=list(dict.fromkeys(t for rows in corpus.values() for r in rows for t in r.values()))
    text_to_index={t:i for i,t in enumerate(texts)}
    cache_contract=dict(data=DATA_ID,teacher=GTE,length=CFG['max_length'],pooling='cls',
                        normalized=True,storage='float32',format_version=1)
    cache_dir=SOURCE_RUN/'gte_cache'/digest(cache_contract)
    assert json.loads((cache_dir/'contract.json').read_text())==cache_contract
    completion=json.loads((cache_dir/'complete.json').read_text())
    assert completion['count']==len(texts) and completion['cache_id']==digest(cache_contract)
    arrays=[]
    for start in tqdm(range(0,len(texts),512),desc='Read existing target cache'):
        subset=texts[start:start+512]
        expected=np.asarray([hashlib.sha256(t.encode()).hexdigest() for t in subset])
        with np.load(cache_dir/f'{start:09d}.npz',allow_pickle=False) as shard:
            assert np.array_equal(shard['ids'],expected), 'Target/text identity mismatch.'
            values=shard['vectors'].copy()
        assert values.shape==(len(subset),768) and np.isfinite(values).all()
        assert np.allclose(np.linalg.norm(values,axis=1),1,atol=1e-4)
        arrays.append(values)
    teacher_targets=torch.from_numpy(np.concatenate(arrays));del arrays
    print('Reused cached teacher targets:',teacher_targets.shape)
else:
    print('Evaluation-only run: no training pairs or teacher targets need to be loaded.')

Evaluation-only run: no training pairs or teacher targets need to be loaded.


In [12]:
# Completed generations are immutable. Upload to a temporary directory; COMMITTED.json is written last.
# Resume discovers the newest verified generation, rather than trusting a possibly stale "latest" pointer.
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), cpu=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all())
def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy'])
    torch.set_rng_state(state['cpu'])
    if torch.cuda.is_available(): torch.cuda.set_rng_state_all(state['cuda'])

def save_checkpoint(stage, step, model, optimizer, scheduler, sampler, best_metric, best_step, total_steps):
    parent=ROOT/'checkpoints'/stage; parent.mkdir(parents=True,exist_ok=True)
    generation=f'{step:08d}-{uuid.uuid4().hex[:8]}'
    local=LOCAL/(generation+'.pt')
    torch.save(dict(stage=stage,step=step,total_steps=total_steps,contract=CONTRACT_ID,data_id=DATA_ID,
                    family=model.family,spec=model.spec,model=model.state_dict(),optimizer=optimizer.state_dict(),
                    scheduler=scheduler.state_dict(),sampler=sampler.state_dict(),rng=rng_state(),
                    best_metric=best_metric,best_step=best_step),local)
    tmp=parent/('.uploading-'+generation); tmp.mkdir()
    shutil.copyfile(local,tmp/'state.pt')
    sha=file_sha(local)
    assert file_sha(tmp/'state.pt') == sha, 'Drive checkpoint copy failed checksum verification'
    write_json(tmp/'COMMITTED.json',dict(step=step,sha256=sha,contract=CONTRACT_ID))
    os.replace(tmp,parent/generation)
    local.unlink()
    # Retain latest two plus the generation containing the best validation result.
    generations=sorted(p for p in parent.iterdir() if (p/'COMMITTED.json').exists() and not p.name.startswith('.'))
    keep={p.name for p in generations[-2:]}
    keep.update(p.name for p in generations if int(p.name.split('-')[0]) == best_step)
    for p in generations:
        if p.name not in keep: shutil.rmtree(p)
    return parent/generation

def latest_checkpoint(stage):
    parent=ROOT/'checkpoints'/stage
    if not parent.exists(): return None
    candidates=sorted((p for p in parent.iterdir() if not p.name.startswith('.') and (p/'COMMITTED.json').exists()),reverse=True)
    for path in candidates:
        try:
            manifest=json.loads((path/'COMMITTED.json').read_text())
            assert manifest['contract']==CONTRACT_ID
            assert file_sha(path/'state.pt')==manifest['sha256']
            return path
        except (OSError,ValueError,AssertionError) as e:
            print('Skipping invalid checkpoint',path,e)
    if candidates: raise RuntimeError('No valid checkpoint remains; inspect Drive files.')
    return None

def load_checkpoint(stage):
    path=latest_checkpoint(stage)
    if path is None: return None
    local=LOCAL/(stage+'-resume.pt'); shutil.copyfile(path/'state.pt',local)
    # Only our own checkpoints, never arbitrary downloaded pickle files.
    state=torch.load(local,map_location='cpu',weights_only=False)
    assert state['contract']==CONTRACT_ID and state['data_id']==DATA_ID
    return state

class StatefulSampler:
    def __init__(self,n,batch,seed):
        assert n>=batch
        self.n,self.batch=n,batch
        self.generator=torch.Generator().manual_seed(seed)
        self.order=torch.randperm(n,generator=self.generator); self.cursor=0; self.epoch=0
    def next(self):
        if self.cursor+self.batch>self.n:
            self.order=torch.randperm(self.n,generator=self.generator); self.cursor=0; self.epoch+=1
        ids=self.order[self.cursor:self.cursor+self.batch].tolist(); self.cursor+=self.batch
        return ids
    def state_dict(self):
        return dict(n=self.n,batch=self.batch,order=self.order,cursor=self.cursor,epoch=self.epoch,generator=self.generator.get_state())
    def load_state_dict(self,s):
        assert (self.n,self.batch)==(s['n'],s['batch'])
        self.order=s['order'];self.cursor=s['cursor'];self.epoch=s['epoch'];self.generator.set_state(s['generator'])

In [13]:
def flatten_pairs(rows):
    # This ordering is used for BOTH students and targets.
    return [r[k] for k in ('query','positive','negative') for r in rows]

def contrastive_loss(z, rows, dim):
    n=len(rows); z=F.normalize(z[:,:dim].float(),dim=-1)
    q,p,neg=z.split(n)
    docs=torch.cat((p,neg))
    logits=q@docs.T/CFG['temperature']
    doc_ids=[text_id(r[k]) for k in ('positive','negative') for r in rows]
    pos_ids=[text_id(r['positive']) for r in rows]
    query_ids=[text_id(r['query']) for r in rows]
    # Exclude duplicated positives and all alternative passages of duplicate queries from negatives.
    valid=torch.ones((n,2*n),device=z.device,dtype=torch.bool)
    for i in range(n):
        for j in range(2*n):
            if j!=i and (doc_ids[j]==pos_ids[i] or (j%n!=i and query_ids[j%n]==query_ids[i])):
                valid[i,j]=False
    logits=logits.masked_fill(~valid,torch.finfo(logits.dtype).min)
    return F.cross_entropy(logits,torch.arange(n,device=z.device))

def objective(model, raw, targets, rows, dimensions=None):
    z=F.normalize(raw.float(),dim=-1); target=F.normalize(targets.float(),dim=-1)
    aligned=F.normalize(model.align(raw).float(),dim=-1)
    align=(1-(aligned*target).sum(-1)).mean()
    pairmask=~torch.eye(len(z),device=z.device,dtype=torch.bool)
    relation=F.mse_loss((z@z.T)[pairmask],(target@target.T)[pairmask])
    dims=dimensions or [model.output_dim]
    contrastive=torch.stack([contrastive_loss(raw,rows,d) for d in dims]).mean()
    loss=CFG['contrastive_weight']*contrastive+CFG['alignment_weight']*align+CFG['relational_weight']*relation
    return loss,dict(contrastive=contrastive.detach().item(),alignment=align.detach().item(),
                     relational=relation.detach().item())

def targets_for(rows, teacher=None):
    batch_texts=flatten_pairs(rows)
    if teacher is None:
        return teacher_targets[[text_to_index[t] for t in batch_texts]].to(DEVICE)
    teacher.eval()
    with torch.no_grad(),torch.autocast('cuda',dtype=torch.bfloat16):
        return F.normalize(teacher(batch_texts).float(),dim=-1)



In [14]:
@torch.inference_mode()
def validate(model,teacher=None,split='validation'):
    assert teacher is None, 'This continuation uses the cached GTE supervisor.'
    model.eval();rows=corpus[split]  # All 256 held-out validation examples, not the pilot's 128 subset.
    queries,positives=[],[];loss_sum=0.0;cos_sum=0.0
    for start in range(0,len(rows),CFG['batch_size']):
        batch=rows[start:start+CFG['batch_size']];n=len(batch);target=targets_for(batch)
        with torch.autocast('cuda',dtype=torch.bfloat16):
            raw=model(flatten_pairs(batch));loss,_=objective(model,raw,target,batch,DIMS)
            aligned=F.normalize(model.align(raw).float(),dim=-1)
        queries.append(raw[:n].float().cpu());positives.append(raw[n:2*n].float().cpu())
        loss_sum+=loss.item()*n;cos_sum+=(aligned*F.normalize(target.float(),dim=-1)).sum().item()/3
    q,p=torch.cat(queries),torch.cat(positives)
    ids=[text_id(r['positive']) for r in rows]
    relevant=torch.tensor([[a==b for b in ids] for a in ids])
    profiles={}
    for dim in DIMS:
        scores=F.normalize(q[:,:dim],dim=-1)@F.normalize(p[:,:dim],dim=-1).T
        best_positive=scores.masked_fill(~relevant,-float('inf')).max(1).values
        # Duplicate positives are relevant; all tied irrelevant passages conservatively outrank them.
        ranks=1+((scores>=best_positive[:,None]-1e-6)&~relevant).sum(1)
        profiles[str(dim)]=dict(hit_at_1=float((ranks==1).float().mean()),mrr=float((1/ranks.float()).mean()))
    result=dict(loss=loss_sum/len(rows),aligned_teacher_cosine=cos_sum/len(rows),examples=len(rows),
                profiles=profiles,selection_score=float(np.mean([x['mrr'] for x in profiles.values()])))
    print(split,result,flush=True)
    return result

In [15]:
def train_stage(model, stage, steps, lr, teacher=None, dimensions=None):
    model.to(DEVICE).requires_grad_(True)
    if teacher is not None:
        assert model is not teacher
        teacher.to(DEVICE).eval().requires_grad_(False)
    optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=CFG['weight_decay'])
    scheduler=get_cosine_schedule_with_warmup(optimizer,int(steps*CFG['warmup_fraction']),steps)
    sampler=StatefulSampler(len(corpus['train']),CFG['batch_size'],CFG['seed'])
    state=load_checkpoint(stage)
    start=0;best_metric=-float('inf');best_step=0
    if state is not None:
        assert state['total_steps']==steps and state['spec']==model.spec
        model.load_state_dict(state['model'],strict=True)
        optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler'])
        sampler.load_state_dict(state['sampler']);restore_rng(state['rng'])
        start=state['step'];best_metric=state['best_metric'];best_step=state['best_step'];del state
        print('Resuming',stage,start,'/',steps,flush=True)
    else:
        initial_metrics=validate(model,teacher)
        best_metric=initial_metrics['selection_score'];best_step=0
        # Keep the unmodified source as a selectable checkpoint if fine-tuning never improves validation.
        save_checkpoint(stage,0,model,optimizer,scheduler,sampler,best_metric,best_step,steps)
    last_step=start
    try:
        for step in range(start+1,steps+1):
            model.train();optimizer.zero_grad(set_to_none=True);running=0.0
            # Save rollback state: an interrupt in an accumulation window must not skip examples.
            boundary_sampler=copy.deepcopy(sampler.state_dict());boundary_rng=rng_state()
            for _ in range(CFG['accumulation']):
                rows=[corpus['train'][i] for i in sampler.next()]
                target=targets_for(rows,teacher)
                with torch.autocast('cuda',dtype=torch.bfloat16):
                    raw=model(flatten_pairs(rows));loss,parts=objective(model,raw,target,rows,dimensions)
                if not torch.isfinite(loss): raise FloatingPointError(f'{stage}: non-finite loss')
                (loss/CFG['accumulation']).backward();running+=loss.item()/CFG['accumulation']
            norm=nn.utils.clip_grad_norm_(model.parameters(),1.0,error_if_nonfinite=True)
            optimizer.step();scheduler.step();last_step=step
            if step%CFG['log_every']==0 or step==1:
                print(f'{stage} {step}/{steps} loss={running:.4f} '+str(parts),flush=True)
            improved=False
            if step%CFG['eval_every']==0 or step==steps:
                metrics=validate(model,teacher)
                with open(ROOT/'metrics.jsonl','a') as f:
                    f.write(json.dumps(dict(stage=stage,step=step,train_loss=running,validation=metrics))+'\n')
                if metrics['selection_score']>best_metric:
                    best_metric=metrics['selection_score'];best_step=step;improved=True
            if step%CFG['checkpoint_every']==0 or step==steps or improved:
                save_checkpoint(stage,step,model,optimizer,scheduler,sampler,best_metric,best_step,steps)
    except KeyboardInterrupt:
        # Discard incomplete gradients and restore sample order for the unfinished optimizer step.
        if 'step' in locals() and last_step<step:
            optimizer.zero_grad(set_to_none=True);sampler.load_state_dict(boundary_sampler);restore_rng(boundary_rng)
        save_checkpoint(stage,last_step,model,optimizer,scheduler,sampler,best_metric,best_step,steps)
        print('Checkpoint saved. Rerun this cell to resume.');raise
    model.eval()
    return model

def restore_best(model,stage):
    state=load_checkpoint(stage)
    assert state is not None
    best=state['best_step']; del state
    parent=ROOT/'checkpoints'/stage
    choices=sorted(p for p in parent.iterdir() if p.name.startswith(f'{best:08d}-') and (p/'COMMITTED.json').exists())
    assert choices, 'Best checkpoint is missing'
    path=choices[-1];meta=json.loads((path/'COMMITTED.json').read_text())
    assert file_sha(path/'state.pt')==meta['sha256']
    state=torch.load(path/'state.pt',map_location='cpu',weights_only=False)
    model.load_state_dict(state['model'],strict=True)
    print('Selected',stage,'validation-best step',best)
    return model

In [16]:
if RUN_MATRYOSHKA:
    student.load_state_dict(baseline_weights,strict=True)
    student=train_stage(student,'matryoshka',REFINEMENT['steps'],REFINEMENT['lr'],dimensions=DIMS)
    student=restore_best(student,'matryoshka')
    selected_state=load_checkpoint('matryoshka')
    selected_step=selected_state['best_step'];del selected_state
    write_json(ROOT/'evaluations'/'matryoshka-selected-test.json',
               dict(selected_step=selected_step,metrics=validate(student,split='test')))
    refinement_results=benchmark(student,'matryoshka-selected-prefixes',dims=DIMS)
    compare_prefixes(refinement_results,reference=baseline_results)
else:
    student.load_state_dict(baseline_weights,strict=True)
    print('Original selected checkpoint retained; no training performed.')

Original selected checkpoint retained; no training performed.


## Export the selected model and verify every prefix
 The export is in the new destination run. Profiles share one backbone; vector truncation does not prune layers.
 To reuse a profile: encode, slice `vector[:dimension]`, then L2-normalize.

In [17]:
# Export inference weights separately from optimizer state; preserve model and tokenizer metadata.
def export_model(model,label):
    assert model.family == 'ettin'
    final=ROOT/'exports'/label
    if final.exists():
        print('Export already exists:',final,'(use another label to export a new version)')
        return final
    temp=ROOT/'exports'/('.uploading-'+uuid.uuid4().hex);temp.mkdir(parents=True)
    model.tokenizer.save_pretrained(temp/'tokenizer')
    if model.family=='ettin':
        model.backbone.save_pretrained(temp/'backbone',safe_serialization=True)
    else:
        save_file({k:v.detach().cpu().contiguous() for k,v in model.backbone.state_dict().items()},str(temp/'backbone.safetensors'))
    write_json(temp/'encoder.json',dict(family=model.family,spec=model.spec,output_dim=model.output_dim,
               pooling='masked_mean_including_special_tokens',normalize=True,max_length=model.max_length,
               contract=contract,training_only_alignment_exported=False))
    # Preserve the source model card and available upstream notices with redistributed weights.
    source = ETTIN
    source_card = hf_hub_download(source['id'], 'README.md', revision=source['revision'])
    shutil.copyfile(source_card, temp/'UPSTREAM_MODEL_CARD.md')
    write_json(temp/'NOTICE.json', dict(source=source, declared_model_license='MIT',
               supervisor=GTE, supervisor_license='Apache-2.0',
               experiment='Fine-tuned encoder; see upstream model card for attribution.',
               license_url='https://opensource.org/license/mit'))
    os.replace(temp,final)
    print('Inference export:',final)
    return final

def load_export(path):
    path=Path(path);meta=json.loads((path/'encoder.json').read_text())
    tok=AutoTokenizer.from_pretrained(path/'tokenizer')
    assert meta['family']=='ettin'
    backbone=AutoModel.from_pretrained(path/'backbone',attn_implementation='sdpa')
    model = EmbeddingModel(backbone,tok,meta['output_dim'],meta['family'],meta['spec'])
    model.max_length = meta['max_length']
    return model.to(DEVICE).eval()

In [18]:
label='matryoshka-selected' if RUN_MATRYOSHKA else 'source-baseline'
export_path=export_model(student,label)
# Same backbone for every profile: truncate FIRST, then normalize.
write_json(export_path/'prefix_profiles.json',dict(dimensions=DIMS,truncate_before_normalizing=True,
           full_dimension=student.output_dim,matryoshka_training_enabled=RUN_MATRYOSHKA))
probe=['How do I reset my password?','Use the password recovery link.','It is raining outside.']
before=encode_texts(student,probe)
reloaded=load_export(export_path);after=encode_texts(reloaded,probe)
for dim in DIMS:
    torch.testing.assert_close(F.normalize(before[:,:dim],dim=-1),F.normalize(after[:,:dim],dim=-1),atol=1e-5,rtol=1e-4)
print('Export/reload parity passed at every dimension:',DIMS)
del reloaded;gc.collect();torch.cuda.empty_cache()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Inference export: /content/drive/MyDrive/encoder_experiments/ettin-prefix-eval-01/exports/source-baseline


Loading weights:   0%|          | 0/62 [00:00<?, ?it/s]

Export/reload parity passed at every dimension: [128, 256, 384]


## Attribution and limits
 Uses the MIT-licensed [Ettin](https://huggingface.co/jhu-clsp/ettin-encoder-32m) checkpoint and cached targets from
 Apache-2.0 [GTE-ModernBERT](https://huggingface.co/Alibaba-NLP/gte-modernbert-base).
 Full provenance and revisions are inherited from the pilot and recorded in this run's config and export.
 This remains an English short-text experiment. Selecting hyperparameters repeatedly against NanoBEIR would turn
 it into development data; keep a separate final benchmark for later claims.

## Final independent sanity check
Run after baseline evaluation or optional refinement. Results are saved in this continuation run.

In [19]:
# Quick independent sanity check -- run this cell after any training stage.
# Select neo_teacher explicitly to check NeoBERT before pruning, or use load_export(path).
SANITY_MODEL = globals().get('student', globals().get('neo_teacher'))
SANITY_LABEL = 'current-model'  # Change to e.g. 'teacher', 'pruned', 'student', 'matryoshka'.
assert SANITY_MODEL is not None, 'Create a model above, or set SANITY_MODEL = load_export(path).'

# Authored probes, separate from the training pairs and NanoBEIR. Do not train on these.
# (query, relevant passage, unrelated passage). A tiny diagnostic, not a quality benchmark.
SANITY_CASES = [
    ('I forgot my account password.', 'Use the password reset link to regain access to your account.',
     'The tomato plants need watering every morning.'),
    ('How can I get my money back for this purchase?', 'Request a refund for your order through customer support.',
     'The train leaves the station at six in the evening.'),
    ('My phone battery runs out too quickly.', 'Reduce screen brightness and background apps to extend battery life.',
     'The museum has opened a new exhibition of ancient pottery.'),
    ('How do I bake a loaf of bread?', 'Mix flour, water and yeast, let the dough rise, then bake it in the oven.',
     'The football team won the championship yesterday.'),
    ('I want to cancel my monthly subscription.', 'Open your billing settings and choose cancel subscription.',
     'Migrating birds travel south during the winter.'),
    ('My laptop cannot connect to Wi-Fi.', 'Check the wireless network settings and restart your router.',
     'The orchestra performed a symphony at the concert hall.'),
    ('How can I track the package I ordered?', 'Enter your parcel tracking number to see its delivery status.',
     'Stretching your legs before a run can help improve flexibility.'),
    ('How do I change the language in the app?', 'Choose your preferred display language in the application settings.',
     'The recipe calls for two eggs and a cup of milk.'),
    ('I need to book a visit with a doctor.', 'Contact the clinic to schedule a medical appointment.',
     'The new telescope can observe distant galaxies.'),
    ('Where can I find books to borrow?', 'Visit the public library to check out books with your library card.',
     'The car needs new tires before the road trip.'),
    ('How can I recover a file that I deleted?', 'Look in the recycle bin or restore the document from a backup.',
     'The beach is covered in shells after the tide goes out.'),
    ('I want to learn to play the guitar.', 'Practice basic chords and strumming patterns on your guitar each day.',
     'The city council approved construction of a new bridge.'),
]

def sanity_scores(raw, cases, dimensions):
    # Pure scoring function: no model, tokenizer or teacher dependency.
    raw = raw.detach().float().cpu()
    n = len(cases)
    assert raw.ndim == 2 and len(raw) == 3*n
    finite = bool(torch.isfinite(raw).all())
    norms = raw.norm(dim=-1)
    nonzero = finite and bool((norms > 1e-8).all())
    result = dict(finite=finite, nonzero=nonzero,
                  raw_norm_min=float(norms.min()) if finite else None,
                  raw_norm_mean=float(norms.mean()) if finite else None,
                  profiles={})
    if not nonzero:
        return result
    for dim in dimensions:
        prefix_norms = raw[:, :dim].norm(dim=-1)
        z = F.normalize(raw[:, :dim], dim=-1)
        q, positive, unrelated = z.split(n)
        pos = (q*positive).sum(-1); neg = (q*unrelated).sum(-1)
        scores = q @ torch.cat((positive, unrelated)).T
        positive_mask = torch.zeros_like(scores, dtype=torch.bool)
        positive_mask[torch.arange(n), torch.arange(n)] = True
        # Conservative tie handling: a collapsed model must not pass by index order.
        ranks = 1 + ((scores >= pos[:,None]-1e-6) & ~positive_mask).sum(-1)
        pairwise = z @ z.T
        offdiag = pairwise[~torch.eye(len(z),dtype=torch.bool)]
        result['profiles'][str(dim)] = dict(
            nonzero_prefix=bool((prefix_norms > 1e-8).all()),
            easy_accuracy=float((pos > neg+1e-6).float().mean()),
            mean_margin=float((pos-neg).mean()),
            hit_at_1=float((ranks <= 1).float().mean()),
            hit_at_5=float((ranks <= 5).float().mean()),
            mrr=float((1/ranks.float()).mean()),
            normalized_spread=float((z-z.mean(0)).square().sum(-1).mean()),
            off_diagonal_cosine_mean=float(offdiag.mean()),
            examples=[dict(query=cases[i][0], positive=cases[i][1], unrelated=cases[i][2],
                           positive_cosine=float(pos[i]), unrelated_cosine=float(neg[i]),
                           positive_rank=int(ranks[i]), easy_correct=bool(pos[i]>neg[i]+1e-6))
                      for i in range(n)])
    return result

@torch.inference_mode()
def run_bottom_sanity(model, label):
    was_training = model.training
    model.eval()
    model_device = next(model.parameters()).device
    def encode_raw(texts):
        outputs = []
        for start in range(0,len(texts),4):
            ctx = torch.autocast('cuda',dtype=torch.bfloat16) if model_device.type == 'cuda' else nullcontext()
            with ctx:
                outputs.append(model(texts[start:start+4]).float().cpu())
        return torch.cat(outputs)
    try:
        # Same order as the scoring function: queries, positives, unrelated passages.
        texts = [row[column] for column in range(3) for row in SANITY_CASES]
        raw = encode_raw(texts)
        dimensions = sorted(set([d for d in (128,256,384) if d < raw.shape[1]] + [raw.shape[1]]))
        report = sanity_scores(raw,SANITY_CASES,dimensions)
        report.update(label=label, family=getattr(model,'family','unknown'),
                      contract=globals().get('CONTRACT_ID'), cases=len(SANITY_CASES),
                      candidate_passages=2*len(SANITY_CASES),
                      note='Small authored diagnostic. Truncated profiles are exploratory unless trained for them.')
        short = SANITY_CASES[0][0]
        long = 'This is a much longer passage about a library, its collection of books, the weather outside, and plans for a journey. '
        a = encode_raw([short]); repeated = encode_raw([short]); padded = encode_raw([short,long])[:1]
        repeat_ok = bool(torch.isfinite(a).all() and torch.isfinite(repeated).all() and
                         torch.allclose(a,repeated,atol=1e-5,rtol=1e-4))
        padding_cosine = float(F.cosine_similarity(a,padded).item()) if bool(torch.isfinite(a).all() and torch.isfinite(padded).all()) else None
        report['repeatable'] = repeat_ok
        report['padding_cosine'] = padding_cosine
        report['plumbing_passed'] = bool(report['finite'] and report['nonzero'] and repeat_ok and
                                         padding_cosine is not None and padding_cosine > .999)
        flags = []
        for dim, metrics in report['profiles'].items():
            if not metrics['nonzero_prefix'] or metrics['normalized_spread'] < 1e-4:
                flags.append(f'{dim}D: zero or nearly collapsed embeddings.')
            if metrics['easy_accuracy'] <= .5:
                flags.append(f'{dim}D: easy semantic ranking at or below 50%; inspect examples.')
        report['flags'] = flags
        path = ROOT/'evaluations'/f'sanity-{time.time_ns()}.json'
        write_json(path,report)
        print('Plumbing:', 'PASS' if report['plumbing_passed'] else 'FAIL',
              '| repeatable:', repeat_ok, '| single/padded cosine:', padding_cosine)
        print(' dim   easy    hit@1   hit@5    margin    spread')
        for dim,metrics in report['profiles'].items():
            print(f"{dim:>4}  {metrics['easy_accuracy']:6.1%}  {metrics['hit_at_1']:6.1%}  "
                  f"{metrics['hit_at_5']:6.1%}  {metrics['mean_margin']:8.4f}  {metrics['normalized_spread']:8.4f}")
        if report['profiles']:
            print('\nFull-vector examples (positive should outrank unrelated):')
            for row in report['profiles'][str(raw.shape[1])]['examples']:
                print(f"{'OK' if row['easy_correct'] else 'MISS':4}  positive={row['positive_cosine']:.3f} "
                      f"unrelated={row['unrelated_cosine']:.3f} rank={row['positive_rank']:2}  {row['query']}")
        for flag in flags: print('Inspect:',flag)
        print('\nSaved to Drive:',path)
        print('Passing the plumbing checks does not establish semantic quality; use the retrieval benchmarks too.')
        if not report['plumbing_passed']:
            raise RuntimeError(f'Numerical, repeatability or padding check failed. Details: {path}')
        return report
    finally:
        model.train(was_training)

sanity_report = run_bottom_sanity(SANITY_MODEL,SANITY_LABEL)

Plumbing: PASS | repeatable: True | single/padded cosine: 0.9999837279319763
 dim   easy    hit@1   hit@5    margin    spread
 128  100.0%   91.7%  100.0%    0.3852    0.5493
 256  100.0%   91.7%  100.0%    0.3824    0.5251
 384  100.0%   91.7%  100.0%    0.3599    0.5016

Full-vector examples (positive should outrank unrelated):
OK    positive=0.857 unrelated=0.370 rank= 1  I forgot my account password.
OK    positive=0.752 unrelated=0.434 rank= 1  How can I get my money back for this purchase?
OK    positive=0.840 unrelated=0.443 rank= 1  My phone battery runs out too quickly.
OK    positive=0.786 unrelated=0.367 rank= 1  How do I bake a loaf of bread?
OK    positive=0.841 unrelated=0.413 rank= 1  I want to cancel my monthly subscription.
OK    positive=0.787 unrelated=0.391 rank= 1  My laptop cannot connect to Wi-Fi.
OK    positive=0.687 unrelated=0.426 rank= 1  How can I track the package I ordered?
OK    positive=0.764 unrelated=0.333 rank= 1  How do I change the language in the a